# ATML PA-4 · Task 1 — Model Compression (pruning + quantization) of Qwen2.5-1.5B

**What this notebook does, end to end:** loads the unchanged FP16 base model once, builds fixed data selections,
then runs every required condition through *one* identical evaluation function:

| Step | Condition(s) | What changes |
|---|---|---|
| 1 | FP16 baseline | nothing — the reference every comparison is paired against |
| 2 | P1 magnitude pruning 25/50/75% | zero the smallest \|W\| in each output row |
| 3 | P2 Wanda 25/50/75% | zero the smallest \|W\|·‖X‖ in each output row |
| 4 | P3 non-uniform Wanda 50/75% | *how much* to prune per block (sensitivity + allocation rule) |
| 8 | Wanda 2:4 | 2 of every 4 consecutive weights kept (structured) |
| 5 | Q1 round-to-nearest INT8/4/3 | group-wise asymmetric weight quantization |
| 6 | Q2 AWQ-style INT8/4/3 | activation-aware channel scales before the same quantizer |
| 7 | Q3 mixed precision | per-block bits ∈ {2,3,4,8} under the Q2 INT4 / INT3 byte budgets |
| 9 | Analysis | tables, paired bootstrap CIs, figures, qualitative cases |

**Kaggle settings:** Accelerator **GPU T4** (one GPU is used) · **Internet ON** · *Save Version → Save & Run All*.
Expected run time ≈ 2.5–3.5 h. Everything is written to `/kaggle/working/t1_outputs/`.

**Reading the output:** every important stage prints `[PASS]/[FAIL]` with an *intended* and an *actual* value.

**If the run stops part-way:** attach this notebook's output as an input (*Add Input → Your Work → this notebook*) and run again —
finished conditions are detected and skipped (`[skip] ...`), and probes/selections are re-loaded from disk.

**Data note:** the course `atml-pa4-datasets` manifests were not available when this was written, so the *Manifests* cell
builds equivalent fixed selections with seed 6304 and saves every ID/offset/window. If you have the official files,
point that cell at them instead (keep the same variable names).

In [ ]:
!pip -q install "transformers==4.57.1" "huggingface_hub==0.36.2"

## 0 · Configuration and helpers
All fixed settings (seed 6304, budgets, group size 128, grids) are in the next cell. The library cells after it only *define*
functions — nothing heavy runs until **Load model**.

In [ ]:
# ============================================================
# CONFIGURATION — every fixed setting of Task 1 lives here
# ============================================================
import os, sys, json, math, time, re, random, gc, platform, glob, shutil, traceback
from pathlib import Path
from collections import OrderedDict, defaultdict
import numpy as np
import torch
import torch.nn.functional as F

SEED = 6304
MODEL_ID = "Qwen/Qwen2.5-1.5B"          # BASE checkpoint (not -Instruct)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16                    # the unchanged model is FP16
OUT_DIR = Path("/kaggle/working/t1_outputs")

PROJS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
ATTN_PROJS = {"q_proj", "k_proj", "v_proj", "o_proj"}
# projections that read the SAME input vector share one activation statistic
GROUPS = OrderedDict([("qkv", ["q_proj", "k_proj", "v_proj"]),
                      ("o", ["o_proj"]),
                      ("gu", ["gate_proj", "up_proj"]),
                      ("down", ["down_proj"])])
GROUP_OF = {p: g for g, ps in GROUPS.items() for p in ps}
HOOK_OF = {"qkv": "q_proj", "o": "o_proj", "gu": "gate_proj", "down": "down_proj"}

CALIB_N, SEQ_LEN, DEV_N = 128, 512, 32   # calibration / development manifests (C4 train)
EVAL_SCORED_TOKENS = 50_000              # per perplexity suite
WIN = 512                                # perplexity window length
HS_N, HS_SUB = 2000, 512                 # HellaSwag full manifest / paired diagnostic subset
GROUP_SIZE = 128                         # quantization group along input columns
PROBE_N = 128                            # calibration seqs used by sensitivity probes (same IDs as Wanda/AWQ)
ERR_N = 16                               # calibration seqs used for layer-output error
AWQ_TOKENS = 4096                        # token rows sampled per layer for AWQ scale search
AWQ_GRID = [round(i / 10, 2) for i in range(11)]   # alpha grid {0, 0.1, ..., 1.0}
P3_LAMBDAS = [0.04, 0.08]                # P3 allocation strength candidates (chosen on DEV only)
OWL_M = 5.0                              # OWL outlier threshold multiplier
SPARSITY_BOUNDS = (0.10, 0.85)
Q3_BITS = [2, 3, 4, 8]
TIMING_REPS = 5
BOOT = 2000
BS_CALIB, BS_PPL, BS_HS = 8, 4, 16

CHECKS = []   # every intended-vs-actual check is appended here and saved at the end

In [ ]:
# ============================================================
# SMALL UTILITIES
# ============================================================
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def check(label, intended, actual, ok=None):
    """Print an INTENDED vs ACTUAL line and remember it."""
    if ok is None:
        ok = (intended == actual)
    tag = "PASS" if ok else "FAIL"
    print(f"[{tag}] {label}\n        intended: {intended}\n        actual:   {actual}")
    CHECKS.append(dict(label=label, intended=str(intended), actual=str(actual), ok=bool(ok)))
    return bool(ok)


def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def to_py(x):
    if isinstance(x, dict):
        return {str(k): to_py(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_py(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if torch.is_tensor(x):
        return x.detach().cpu().tolist()
    return x


def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(to_py(obj), f, indent=1)


def load_json(path):
    with open(path) as f:
        return json.load(f)


def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def mb(nbytes):
    return nbytes / 2**20

### Model helpers + the block runner
*Eligible* matrices are the 7 projections in each of the 28 blocks (`q,k,v,o,gate,up,down`); embeddings, norms, biases and the
(tied) LM head are never touched. The **block runner** pushes calibration hidden states through the network one block at a time,
which is what Wanda (and AWQ's statistics) need: collect block ℓ's inputs → modify block ℓ → feed its outputs to block ℓ+1.

In [ ]:
# ============================================================
# MODEL HELPERS: eligible matrices, parameter accounting, snapshot/restore
# ============================================================
def get_proj(layer, p):
    return getattr(layer.self_attn if p in ATTN_PROJS else layer.mlp, p)


def eligible_linears(model):
    """OrderedDict 'layer.proj' -> nn.Linear for the 7 eligible projections of every block."""
    out = OrderedDict()
    for i, layer in enumerate(model.model.layers):
        for p in PROJS:
            out[f"{i}.{p}"] = get_proj(layer, p)
    return out


def elig_shapes(model):
    return OrderedDict((n, tuple(m.weight.shape)) for n, m in eligible_linears(model).items())


def param_accounting(model):
    """Count parameters ONCE per storage (tied embedding/lm_head counted once)."""
    seen = OrderedDict()
    for n, p in model.named_parameters(remove_duplicate=False):
        seen.setdefault(p.data_ptr(), (n, p))
    n_total = sum(p.numel() for _, p in seen.values())
    elig = eligible_linears(model)
    n_elig = sum(m.weight.numel() for m in elig.values())
    tied = model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr()
    return dict(n_total_unique=n_total, n_eligible=n_elig, n_noneligible=n_total - n_elig,
                n_eligible_matrices=len(elig), lm_head_tied_to_embeddings=bool(tied),
                n_layers=len(model.model.layers))


def snapshot(model):
    """CPU copy of every (unique) parameter — the unchanged FP16 checkpoint."""
    return {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters()}


@torch.no_grad()
def restore_(model, snap, prefix=None):
    for n, p in model.named_parameters():
        if prefix is None or n.startswith(prefix):
            p.copy_(snap[n].to(p.device))


@torch.no_grad()
def get_eligible_weights_cpu(model):
    return {n: m.weight.detach().to("cpu", copy=True) for n, m in eligible_linears(model).items()}


@torch.no_grad()
def apply_weights_(model, W):
    for n, m in eligible_linears(model).items():
        if n in W:
            m.weight.copy_(W[n].to(m.weight.device, m.weight.dtype))

In [ ]:
# ============================================================
# BLOCK RUNNER: push calibration hidden states through the network ONE BLOCK AT A TIME.
# This lets us (a) collect the inputs of block l, (b) modify block l, and
# (c) feed block l's (modified) outputs to block l+1 — without re-running the whole model.
# ============================================================
@torch.no_grad()
def embed_batches(model, seqs, bs=BS_CALIB):
    dev = model.model.embed_tokens.weight.device
    batches = []
    for i in range(0, len(seqs), bs):
        ids = torch.tensor(seqs[i:i + bs], device=dev)
        h = model.model.embed_tokens(ids)
        pos = torch.arange(ids.shape[1], device=dev).unsqueeze(0).expand(ids.shape[0], -1)
        pe = model.model.rotary_emb(h, pos)           # (cos, sin) rotary tables
        batches.append([h, pos, pe])
    return batches


@torch.no_grad()
def run_layer(layer, h, pos, pe):
    # attention_mask=None + SDPA => plain causal attention (no padding in calibration data)
    out = layer(h, attention_mask=None, position_ids=pos, position_embeddings=pe, use_cache=False)
    return out[0] if isinstance(out, (tuple, list)) else out


def _group_hooks(layer, fn_by_group):
    """Register forward-pre-hooks on the first projection of each input-sharing group."""
    hs = []
    for g, fn in fn_by_group.items():
        mod = get_proj(layer, HOOK_OF[g])
        hs.append(mod.register_forward_pre_hook(lambda m, inp, _fn=fn: _fn(inp[0])))
    return hs


@torch.no_grad()
def dense_input_stats(model, seqs, bs=BS_CALIB, n_sample=0, seed=SEED, layer_callback=None):
    """One pass over the (unchanged) model. Per block and input group returns
    sq  = sum of x_j^2 over all calibration tokens   (Wanda: ||X_j||_2 = sqrt(sq))
    abs = mean |x_j| over all calibration tokens      (AWQ activation statistic)
    Optionally keeps n_sample fixed random token rows per group and calls
    layer_callback(li, layer, samples, stats_li) BEFORE moving on (used by AWQ search)."""
    H = embed_batches(model, seqs, bs)
    n_tok = len(seqs) * len(seqs[0])
    rng = np.random.default_rng(seed)
    sample_idx = np.sort(rng.choice(n_tok, n_sample, replace=False)) if n_sample else None
    stats = {}
    for li, layer in enumerate(model.model.layers):
        acc = {g: dict(sq=0., abs=0.) for g in GROUPS}
        samples = {g: [] for g in GROUPS}
        state = dict(offset=0)

        def make(g):
            def fn(x):
                x2 = x.reshape(-1, x.shape[-1]).float()
                acc[g]["sq"] = acc[g]["sq"] + x2.pow(2).sum(0)
                acc[g]["abs"] = acc[g]["abs"] + x2.abs().sum(0)
                if sample_idx is not None:
                    lo, hi = state["offset"], state["offset"] + x2.shape[0]
                    sel = sample_idx[(sample_idx >= lo) & (sample_idx < hi)] - lo
                    if len(sel):
                        samples[g].append(x.reshape(-1, x.shape[-1])[torch.as_tensor(sel, device=x.device)].clone())
            return fn
        hooks = _group_hooks(layer, {g: make(g) for g in GROUPS})
        outs = []
        for b in H:
            outs.append(run_layer(layer, *b))
            state["offset"] += b[0].shape[0] * b[0].shape[1]
        for h in hooks:
            h.remove()
        stats[li] = {g: dict(sq=acc[g]["sq"].detach(), absmean=(acc[g]["abs"] / n_tok).detach()) for g in GROUPS}
        if layer_callback is not None:
            smp = {g: torch.cat(samples[g], 0) for g in GROUPS} if sample_idx is not None else None
            layer_callback(li, layer, smp, stats[li])
            del smp
        for b, o in zip(H, outs):
            b[0] = o
        del outs
    del H; free()
    return stats

### Evaluation
* **Perplexity**: each saved window (≤ 512 tokens, from one document) is scored independently; the first token is never scored;
  we keep per-window NLL sums so bootstrap CIs can resample whole windows.
* **HellaSwag**: lm-eval-harness formatting; `acc` = argmax of summed log-likelihood, `acc_norm` = argmax of log-likelihood ÷ number
  of *characters* in the ending.

In [ ]:
# ============================================================
# EVALUATION: perplexity suites, HellaSwag, calibration NLL
# ============================================================
@torch.no_grad()
def seq_nll(model, seqs, bs=BS_PPL):
    """Per-sequence (sum NLL, #scored tokens). The first token of every sequence is never scored."""
    dev = model.model.embed_tokens.weight.device
    sums, counts, nonfinite = [], [], 0
    for i in range(0, len(seqs), bs):
        chunk = seqs[i:i + bs]
        L = max(len(s) for s in chunk)
        ids = torch.zeros(len(chunk), L, dtype=torch.long, device=dev)
        am = torch.zeros(len(chunk), L, dtype=torch.long, device=dev)
        for r, s in enumerate(chunk):
            ids[r, :len(s)] = torch.tensor(s, device=dev); am[r, :len(s)] = 1
        logits = model(input_ids=ids, attention_mask=am, use_cache=False).logits
        for r, s in enumerate(chunk):
            n = len(s)
            lp = F.cross_entropy(logits[r, :n - 1].float(), ids[r, 1:n], reduction="none")
            nonfinite += int((~torch.isfinite(lp)).sum())
            sums.append(float(lp.double().sum())); counts.append(n - 1)
        del logits
    return np.array(sums), np.array(counts), nonfinite


def ppl_summary(sums, counts, nonfinite=0):
    N = int(counts.sum()); mean_nll = float(sums.sum() / N)
    return dict(N=N, mean_nll=mean_nll, ppl=float(math.exp(mean_nll)) if mean_nll < 700 else float("inf"),
                n_windows=len(sums), nonfinite_tokens=int(nonfinite))


def eval_ppl(model, windows):
    sums, counts, nf = seq_nll(model, windows)
    return ppl_summary(sums, counts, nf), sums, counts


def calib_nll(model, seqs):
    sums, counts, _ = seq_nll(model, seqs)
    return float(sums.sum() / counts.sum())


# ---------- HellaSwag (lm-evaluation-harness `hellaswag` formatting) ----------
def hs_preprocess(text):
    text = text.strip()
    text = text.replace(" [title]", ". ")
    text = re.sub("\\[.*?\\]", "", text)
    text = text.replace("  ", " ")
    return text


def hs_format(doc):
    ctx = doc["ctx_a"] + " " + doc["ctx_b"].capitalize()
    query = hs_preprocess(doc["activity_label"] + ": " + ctx)
    choices = [hs_preprocess(e) for e in doc["endings"]]
    return query, choices, int(doc["label"])


def hs_encode(tok, docs):
    """lm-eval style pair encoding: continuation = ' ' + ending, tokens = tok(ctx+cont)[len(tok(ctx)):]."""
    enc = []
    for d in docs:
        query, choices, label = hs_format(d)
        ctx_ids = tok(query, add_special_tokens=False).input_ids
        items = []
        for ch in choices:
            whole = tok(query + " " + ch, add_special_tokens=False).input_ids
            cont = whole[len(ctx_ids):]
            items.append(dict(ctx=ctx_ids, cont=cont, nchar=len(ch)))
        enc.append(dict(id=d["ind"], label=label, items=items, query=query, choices=choices))
    return enc


@torch.no_grad()
def eval_hellaswag(model, enc, bs=BS_HS):
    dev = model.model.embed_tokens.weight.device
    flat = [(ei, ci, it) for ei, e in enumerate(enc) for ci, it in enumerate(e["items"])]
    ll = np.zeros((len(enc), 4)); nchar = np.zeros((len(enc), 4))
    for i in range(0, len(flat), bs):
        chunk = flat[i:i + bs]
        seqs = [it["ctx"] + it["cont"] for _, _, it in chunk]
        L = max(len(s) for s in seqs)
        ids = torch.zeros(len(seqs), L, dtype=torch.long, device=dev)
        am = torch.zeros_like(ids)
        for r, s in enumerate(seqs):
            ids[r, :len(s)] = torch.tensor(s, device=dev); am[r, :len(s)] = 1
        logits = model(input_ids=ids, attention_mask=am, use_cache=False).logits
        for r, (ei, ci, it) in enumerate(chunk):
            c0, n = len(it["ctx"]), len(seqs[r])
            lp = F.log_softmax(logits[r, c0 - 1:n - 1].float(), -1)
            tgt = ids[r, c0:n]
            ll[ei, ci] = float(lp.gather(1, tgt[:, None]).sum())
            nchar[ei, ci] = it["nchar"]
        del logits
    labels = np.array([e["label"] for e in enc])
    pred = ll.argmax(1); pred_n = (ll / nchar).argmax(1)
    return dict(acc=float((pred == labels).mean()), acc_norm=float((pred_n == labels).mean()), n=len(enc)), \
        dict(ll=ll, nchar=nchar, label=labels, ids=np.array([e["id"] for e in enc]))

In [ ]:
# ============================================================
# RESOURCES: storage formulas, nonzeros, latency, peak memory
# ============================================================
def quant_tensor_bytes(dout, din, bits, group=GROUP_SIZE):
    """Packed b-bit payload (bitstream, padded to a byte per tensor) + per-group FP16 scale (2 B)
    + per-group uint8 zero-point (1 B)."""
    n = dout * din
    assert din % group == 0
    return math.ceil(n * bits / 8) + (n // group) * 3


def sparse16_tensor_bytes(n, nnz):
    """Mask format: 1 mask bit per eligible weight (byte padded) + 16 bits per nonzero."""
    return math.ceil(n / 8) + 2 * nnz


def nonzero_counts(model):
    return {n: int((m.weight != 0).sum()) for n, m in eligible_linears(model).items()}


def storage_report(acc, shapes, kind, nnz=None, bits=None, awq_o_scale_bytes=0):
    n_elig = acc["n_eligible"]
    non_elig_bytes = acc["n_noneligible"] * 2           # FP16 embeddings (tied: once), norms, biases
    dense = n_elig * 2
    rep = dict(kind=kind, noneligible_fp16_bytes=non_elig_bytes, eligible_dense_fp16_bytes=dense)
    if nnz is not None:
        tot_nnz = sum(nnz.values())
        rep.update(eligible_nonzeros=tot_nnz, eligible_nonzero_fraction=tot_nnz / n_elig,
                   achieved_sparsity=1 - tot_nnz / n_elig,
                   eligible_sparse16_bytes=sum(sparse16_tensor_bytes(shapes[k][0] * shapes[k][1], nnz[k]) for k in shapes))
    if kind == "dense":
        rep["eligible_bytes"] = dense; rep["nominal_bits_per_weight"] = 16.0
    elif kind == "sparse":
        rep["eligible_bytes"] = rep["eligible_sparse16_bytes"]
        rep["dense_with_zeros_bytes"] = dense
        rep["nominal_bits_per_weight"] = 16.0
    elif kind == "quant":
        qb = sum(quant_tensor_bytes(shapes[k][0], shapes[k][1], bits[k]) for k in shapes)
        rep["eligible_bytes"] = qb + awq_o_scale_bytes
        rep["awq_unfolded_scale_bytes"] = awq_o_scale_bytes
        rep["nominal_bits_per_weight"] = sum(bits[k] * shapes[k][0] * shapes[k][1] for k in shapes) / n_elig
    rep["effective_bits_per_weight_incl_metadata"] = rep["eligible_bytes"] * 8 / n_elig
    rep["total_model_bytes"] = rep["eligible_bytes"] + non_elig_bytes
    rep["total_model_MB"] = mb(rep["total_model_bytes"])
    return rep


def _stats(ts):
    a = np.array(ts)
    return dict(mean=float(a.mean()), std=float(a.std()), min=float(a.min()), reps=len(a), all=a.tolist())


@torch.no_grad()
def measure_latency(model, prompt_ids, reps=TIMING_REPS, decode_tokens=64):
    """Dense FP16 execution path (fake-quantized / zeroed weights still run dense matmuls)."""
    dev = model.model.embed_tokens.weight.device
    cuda = torch.cuda.is_available()
    res = dict(unit="ms", reps=reps, note="dense FP16 kernels; no sparse/low-bit kernels used")
    for L in (512, 2048):
        ids = torch.tensor([prompt_ids[:L]], device=dev)
        for _ in range(2):
            model(input_ids=ids, use_cache=True, logits_to_keep=1)
        sync()
        if cuda: torch.cuda.reset_peak_memory_stats()
        ts = []
        for _ in range(reps):
            sync(); t = time.perf_counter()
            model(input_ids=ids, use_cache=True, logits_to_keep=1)
            sync(); ts.append(1000 * (time.perf_counter() - t))
        res[f"prefill_{L}"] = _stats(ts)
        if cuda: res[f"peak_alloc_MB_prefill_{L}"] = mb(torch.cuda.max_memory_allocated())

    ids = torch.tensor([prompt_ids[:512]], device=dev)

    def decode_once():
        out = model(input_ids=ids, use_cache=True, logits_to_keep=1)
        past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
        sync(); t = time.perf_counter()
        for _ in range(decode_tokens):
            out = model(input_ids=nxt, past_key_values=past, use_cache=True)
            past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
        sync()
        return 1000 * (time.perf_counter() - t)
    decode_once()
    if cuda: torch.cuda.reset_peak_memory_stats()
    ts = [decode_once() for _ in range(reps)]
    res[f"decode_{decode_tokens}_after_512"] = _stats(ts)
    res["decode_ms_per_token"] = float(np.mean(ts) / decode_tokens)
    if cuda: res["peak_alloc_MB_decode"] = mb(torch.cuda.max_memory_allocated())
    return res

### Pruning and allocation functions
`row_mask` zeroes the *k* lowest-scoring weights in every output row (the same convention for magnitude and Wanda).
`k_rows_from_block_sparsity` turns block sparsities into exact per-row zero counts and then fixes the global total **exactly**.

In [ ]:
# ============================================================
# PRUNING: per-row masks, magnitude, sequential Wanda, 2:4
# ============================================================
def row_mask(score, k_rows):
    """True where a weight is PRUNED: the k_rows[r] smallest scores in each output row r."""
    dout, din = score.shape
    if isinstance(k_rows, int):
        k_rows = torch.full((dout,), k_rows, dtype=torch.long, device=score.device)
    k_rows = k_rows.to(score.device)
    order = torch.sort(score, dim=1, stable=True).indices          # ascending score
    rank_keep = torch.arange(din, device=score.device)[None, :] < k_rows[:, None]
    return torch.zeros_like(rank_keep).scatter_(1, order, rank_keep)


def mask_24(score):
    """2:4 — in every group of 4 contiguous input columns prune the 2 lowest scores."""
    dout, din = score.shape
    g = score.reshape(dout, din // 4, 4)
    idx = torch.topk(g, 2, dim=-1, largest=False).indices
    return torch.zeros_like(g, dtype=torch.bool).scatter_(-1, idx, True).reshape(dout, din)


def uniform_k(S):
    return lambda li, p, dout, din: int(round(S * din))


@torch.no_grad()
def magnitude_prune_(model, k_fn):
    for name, m in eligible_linears(model).items():
        li, p = int(name.split(".")[0]), name.split(".")[1]
        W = m.weight
        W[row_mask(W.abs().float(), k_fn(li, p, *W.shape))] = 0


def wanda_score(W, sq):
    return W.abs().float() * sq.sqrt()[None, :]


def score_stats(W, score, mask, sq):
    """Diagnostics kept for RQ1 / P3: Wanda-vs-magnitude mask agreement, activation outliers, outlier ratio."""
    k = mask.sum(1)
    mag_mask = row_mask(W.abs().float(), k)
    norm = sq.sqrt()
    mean_s = score.mean()
    return dict(mask_overlap_with_magnitude=float((mask & mag_mask).sum() / mask.sum().clamp(min=1)),
                act_norm_max_over_median=float(norm.max() / norm.median().clamp(min=1e-12)),
                outlier_ratio_M3=float((score > 3 * mean_s).float().mean()),
                outlier_ratio_M5=float((score > 5 * mean_s).float().mean()),
                outlier_ratio_M7=float((score > 7 * mean_s).float().mean()),
                score_quantiles=[float(v) for v in torch.quantile(score.flatten()[::max(1, score.numel() // 200000)],
                                                                  torch.tensor([.01, .1, .25, .5, .75, .9, .99], device=score.device))])


@torch.no_grad()
def wanda_prune_sequential_(model, seqs, k_fn=None, pattern="unstructured", record=None, bs=BS_CALIB):
    """Wanda as in the official code: block by block, the block's inputs come from the
    ALREADY-PRUNED earlier blocks; all 7 projections of a block are scored from one pass."""
    H = embed_batches(model, seqs, bs)
    for li, layer in enumerate(model.model.layers):
        sq = {g: 0. for g in GROUPS}

        def make(g):
            def fn(x):
                sq[g] = sq[g] + x.reshape(-1, x.shape[-1]).float().pow(2).sum(0)
            return fn
        hooks = _group_hooks(layer, {g: make(g) for g in GROUPS})
        for b in H:
            run_layer(layer, *b)
        for h in hooks:
            h.remove()
        for p in PROJS:
            W = get_proj(layer, p).weight
            score = wanda_score(W, sq[GROUP_OF[p]])
            mask = mask_24(score) if pattern == "2:4" else row_mask(score, k_fn(li, p, *W.shape))
            if record is not None:
                record[f"{li}.{p}"] = score_stats(W, score, mask, sq[GROUP_OF[p]])
            W[mask] = 0
            del score, mask
        for b in H:
            b[0] = run_layer(layer, *b)
    del H; free()


@torch.no_grad()
def wanda_prune_block_dense_(model, li, S, dense_stats):
    """Isolated-removal probe: prune ONE block with Wanda using unchanged-model activation norms."""
    layer = model.model.layers[li]
    for p in PROJS:
        W = get_proj(layer, p).weight
        W[row_mask(wanda_score(W, dense_stats[li][GROUP_OF[p]]["sq"]), int(round(S * W.shape[1])))] = 0

In [ ]:
# ============================================================
# P3 ALLOCATION: sensitivity -> block sparsities -> exact per-row zero counts
# ============================================================
def owl_lod(model, dense_stats, M=OWL_M):
    """OWL layerwise outlier distribution: per block, fraction of pooled Wanda scores > M * mean."""
    lod = []
    for li, layer in enumerate(model.model.layers):
        scores = [wanda_score(get_proj(layer, p).weight, dense_stats[li][GROUP_OF[p]]["sq"]).flatten() for p in PROJS]
        s = torch.cat(scores)
        lod.append(float((s > M * s.mean()).float().mean()))
        del scores, s
    return np.array(lod)


def normalize_importance(r, mode):
    """Map an importance vector to u in [-1, 1]; +1 = most sensitive block."""
    r = np.asarray(r, dtype=float)
    if mode == "minmax":                       # OWL's normalisation
        return 2 * (r - r.min()) / max(r.max() - r.min(), 1e-12) - 1
    ranks = r.argsort().argsort()              # robust rank normalisation (for heavy-tailed probes)
    return 2 * ranks / (len(r) - 1) - 1


def allocate_sparsity(r, S, lam, mode, n_params, lo=SPARSITY_BOUNDS[0], hi=SPARSITY_BOUNDS[1]):
    """s_l = S - lam * u_l, then a common shift delta (bisection) so that the parameter-weighted
    mean sparsity equals S exactly while every s_l stays inside [lo, hi]."""
    u = normalize_importance(r, mode)
    s0 = S - lam * u
    n = np.asarray(n_params, dtype=float)
    f = lambda d: (np.clip(s0 + d, lo, hi) * n).sum() - S * n.sum()
    a, b = -1.0, 1.0
    for _ in range(200):
        mid = (a + b) / 2
        if f(mid) > 0: b = mid
        else: a = mid
    return np.clip(s0 + (a + b) / 2, lo, hi)


def k_rows_from_block_sparsity(s_blocks, shapes, target_zeros):
    """Per-row zero counts realising block sparsities s_l, then fixing the global count EXACTLY."""
    K, total = {}, 0
    for name, (dout, din) in shapes.items():
        s = float(s_blocks[int(name.split(".")[0])])
        z = s * dout * din
        base = int(math.floor(s * din))
        extra = int(round(z - base * dout))           # rows that get one extra zero
        k = torch.full((dout,), base, dtype=torch.long)
        k[:extra] += 1
        K[name] = k; total += int(k.sum())
    D = int(target_zeros) - total
    for name in sorted(shapes, key=lambda n: -shapes[n][0] * shapes[n][1]):
        if D == 0:
            break
        dout, din = shapes[name]
        k = K[name]
        step = 1 if D > 0 else -1
        ok = (k + step >= 0) & (k + step <= din)
        rows = torch.nonzero(ok).flatten()[:abs(D)]
        k[rows] += step; D -= step * len(rows)
    assert D == 0, "could not match the zero budget exactly"
    return K

### Quantization functions
One quantizer for Q1/Q2/Q3: asymmetric, groups of 128 input columns, FP16 scale + uint8 zero-point per group, round-half-to-even,
no clipping search. Packed size = ⌈n·b/8⌉ bytes (LSB-first bitstream) + 3 bytes per group. AWQ only changes *what is fed into*
this quantizer (W·s), and folds 1/s into the previous norm/linear where that is algebraically valid.

In [ ]:
# ============================================================
# QUANTIZATION: group-wise asymmetric uniform quantizer, packing, AWQ
# ============================================================
def fake_quant(W, bits, group=GROUP_SIZE):
    """Asymmetric b-bit, groups of `group` consecutive INPUT columns per output row.
    scale = (wmax - wmin) / (2^b - 1) stored in FP16; zero = clamp(round(-wmin/scale), 0, 2^b-1) stored uint8;
    q = clamp(round(w/scale) + zero, 0, 2^b-1) (round-half-to-even); w_hat = scale * (q - zero).
    No clipping search. Returns (w_hat fp32, q uint8, scale fp16, zero uint8)."""
    dout, din = W.shape
    qmax = 2 ** bits - 1
    Wg = W.float().reshape(dout, din // group, group)
    wmin, wmax = Wg.amin(-1, keepdim=True), Wg.amax(-1, keepdim=True)
    scale = ((wmax - wmin) / qmax).to(torch.float16).float().clamp(min=2 ** -24)
    zero = torch.clamp(torch.round(-wmin / scale), 0, qmax)
    q = torch.clamp(torch.round(Wg / scale) + zero, 0, qmax)
    W_hat = (q - zero) * scale
    return (W_hat.reshape(dout, din), q.to(torch.uint8).reshape(dout, din),
            scale.to(torch.float16).reshape(dout, -1), zero.to(torch.uint8).reshape(dout, -1))


def pack_bits(q, bits):
    """LSB-first bitstream packing of a flat array of b-bit codes, padded to a whole byte."""
    q = np.asarray(q, dtype=np.uint8).reshape(-1)
    bitsarr = np.unpackbits(q[:, None], axis=1, bitorder="little")[:, :bits]
    return np.packbits(bitsarr.reshape(-1), bitorder="little")


def unpack_bits(packed, bits, n):
    b = np.unpackbits(packed, bitorder="little")[:n * bits].reshape(n, bits).astype(np.uint16)
    return (b * (1 << np.arange(bits, dtype=np.uint16))).sum(1).astype(np.uint8)


@torch.no_grad()
def q1_quantize_(model, bits_fn):
    for name, m in eligible_linears(model).items():
        li = int(name.split(".")[0])
        m.weight.copy_(fake_quant(m.weight, bits_fn(li))[0].to(m.weight.dtype))


def awq_scale(absmean, alpha):
    s = absmean.float().pow(alpha).clamp(min=1e-4)
    return s / (s.max() * s.min()).sqrt()


def _mlp_out(layer, X, Wg, Wu, Wd):
    return F.linear(layer.mlp.act_fn(F.linear(X, Wg)) * F.linear(X, Wu), Wd)


@torch.no_grad()
def awq_search_layer(layer, samples, stats_li, bits_list, grid=AWQ_GRID):
    """For each input group and bit width pick alpha minimising calibration output MSE of
    Q(W*s)/s versus W (qkv / o / down: the projections' own outputs; gate+up: the MLP output)."""
    W = {p: get_proj(layer, p).weight.float() for p in PROJS}
    res = {b: {} for b in bits_list}
    for g, ps in GROUPS.items():
        X = samples[g].float()
        if g == "gu":
            ref = _mlp_out(layer, X, W["gate_proj"], W["up_proj"], W["down_proj"])
        else:
            ref = torch.cat([F.linear(X, W[p]) for p in ps], -1)
        ref_energy = float(ref.pow(2).mean())
        for b in bits_list:
            losses = []
            for a in grid:
                s = awq_scale(stats_li[g]["absmean"], a)
                Wq = {p: fake_quant(W[p] * s[None, :], b)[0] / s[None, :] for p in ps}
                if g == "gu":
                    out = _mlp_out(layer, X, Wq["gate_proj"], Wq["up_proj"], W["down_proj"])
                else:
                    out = torch.cat([F.linear(X, Wq[p]) for p in ps], -1)
                losses.append(float((out - ref).pow(2).mean()))
                del Wq, out
            ia = int(np.argmin(losses))
            res[b][g] = dict(alpha=grid[ia], scale=awq_scale(stats_li[g]["absmean"], grid[ia]).half().cpu(),
                             losses=losses, rel_err_best=losses[ia] / ref_energy, rel_err_alpha0=losses[0] / ref_energy)
        del X, ref
    return res


@torch.no_grad()
def apply_awq_block_(layer, scales, bits, quantize=True):
    """Fold inverse scales where algebraically valid, then apply the Q1 quantizer.
    input_layernorm  <- 1/s_qkv   (q,k,v columns *= s_qkv)
    post_attn_norm   <- 1/s_gu    (gate,up columns *= s_gu)
    up_proj rows     <- 1/s_down  (down columns *= s_down)    [h = act(gate) * up is elementwise]
    o_proj: GQA => v_proj has fewer output channels than o_proj inputs, so 1/s_o is NOT folded;
            it is kept as serialized FP16 channel scales (x / s_o at run time), counted in storage."""
    dev = layer.input_layernorm.weight.device
    s = {g: scales[g].to(dev).float() for g in GROUPS}
    ln, pln = layer.input_layernorm.weight, layer.post_attention_layernorm.weight
    ln.copy_((ln.float() / s["qkv"]).to(ln.dtype))
    pln.copy_((pln.float() / s["gu"]).to(pln.dtype))
    qf = (lambda W: fake_quant(W, bits)[0]) if quantize else (lambda W: W)
    for p in ["q_proj", "k_proj", "v_proj"]:
        m = get_proj(layer, p); m.weight.copy_(qf(m.weight.float() * s["qkv"][None, :]).to(m.weight.dtype))
    m = get_proj(layer, "gate_proj"); m.weight.copy_(qf(m.weight.float() * s["gu"][None, :]).to(m.weight.dtype))
    m = get_proj(layer, "up_proj")
    m.weight.copy_(qf(m.weight.float() * s["gu"][None, :] / s["down"][:, None]).to(m.weight.dtype))
    m = get_proj(layer, "down_proj"); m.weight.copy_(qf(m.weight.float() * s["down"][None, :]).to(m.weight.dtype))
    m = get_proj(layer, "o_proj")     # serialized compensation: equivalent to x/s_o then Q(W_o*s_o)
    m.weight.copy_((qf(m.weight.float() * s["o"][None, :]) / s["o"][None, :]).to(m.weight.dtype))


def awq_o_scale_bytes(model):
    return sum(get_proj(l, "o_proj").in_features * 2 for l in model.model.layers)


def awq_effective_weight(W, s, bits):
    return fake_quant(W.float() * s[None, :], bits)[0] / s[None, :]


def dp_bits(cost, budget_bits, bits=Q3_BITS):
    """Exact multiple-choice knapsack: minimise sum_l cost[l][b_l] s.t. sum_l b_l <= budget_bits.
    (Valid as a byte constraint because every block has identical shapes and metadata bytes.)"""
    L = len(cost); INF = float("inf")
    dp = np.full((L + 1, budget_bits + 1), INF); dp[0, 0] = 0
    ch = np.zeros((L + 1, budget_bits + 1), dtype=int)
    for l in range(L):
        for u in range(budget_bits + 1):
            if dp[l, u] == INF: continue
            for b in bits:
                if u + b <= budget_bits and dp[l, u] + cost[l][b] < dp[l + 1, u + b]:
                    dp[l + 1, u + b] = dp[l, u] + cost[l][b]; ch[l + 1, u + b] = b
    u = int(np.argmin(dp[L])); out = []
    for l in range(L, 0, -1):
        b = ch[l, u]; out.append(int(b)); u -= b
    return out[::-1]

### Calibration error + bootstrap
Layer-output error is always measured on **unchanged-model** calibration activations X:
relative error = Σ‖X(W_eff − W)ᵀ‖² / Σ‖XWᵀ‖², so pruning and quantization are measured on the same footing.

In [ ]:
# ============================================================
# CALIBRATION ERROR: per-projection output error on UNCHANGED-model calibration activations
#   rel_err(proj) = sum ||X (W_eff - W)^T||^2 / sum ||X W^T||^2
# plus end-to-end calibration NLL and final-hidden-state error of the compressed model.
# ============================================================
@torch.no_grad()
def local_errors(model, weff_fn, seqs, bs=4):
    """model must currently hold the UNCHANGED weights. weff_fn(name, W_gpu) -> effective weight."""
    H = embed_batches(model, seqs, bs)
    out = {}
    for li, layer in enumerate(model.model.layers):
        W = {p: get_proj(layer, p).weight.float() for p in PROJS}
        D = {p: weff_fn(f"{li}.{p}", get_proj(layer, p).weight).float() - W[p] for p in PROJS}
        num = defaultdict(float); den = defaultdict(float)

        def make(g):
            def fn(x):
                x2 = x.reshape(-1, x.shape[-1]).float()
                for p in GROUPS[g]:
                    num[p] += float(F.linear(x2, D[p]).pow(2).sum())
                    den[p] += float(F.linear(x2, W[p]).pow(2).sum())
            return fn
        hooks = _group_hooks(layer, {g: make(g) for g in GROUPS})
        for b in H:
            b[0] = run_layer(layer, *b)
        for h in hooks:
            h.remove()
        for p in PROJS:
            out[f"{li}.{p}"] = num[p] / max(den[p], 1e-30)
        del W, D
    del H; free()
    return out


@torch.no_grad()
def final_hidden(model, seqs, bs=4):
    dev = model.model.embed_tokens.weight.device
    hs = []
    for i in range(0, len(seqs), bs):
        hs.append(model.model(input_ids=torch.tensor(seqs[i:i + bs], device=dev), use_cache=False).last_hidden_state.cpu())
    return torch.cat(hs)


@torch.no_grad()
def calib_metrics(model, err_seqs, dense_ref):
    nll = calib_nll(model, err_seqs)
    h = final_hidden(model, err_seqs)
    rel = float((h.float() - dense_ref["hidden"].float()).pow(2).sum() / dense_ref["hidden"].float().pow(2).sum())
    return dict(calib_nll=nll, calib_delta_nll=nll - dense_ref["nll"], final_hidden_rel_err=rel)

In [ ]:
# ============================================================
# PAIRED BOOTSTRAP (95% percentile intervals)
# ============================================================
def boot_ppl_diff(sA, nA, sB, nB, B=BOOT, seed=SEED):
    """PPL(A) - PPL(B): resample whole windows, pool token NLL, then exponentiate."""
    rng = np.random.default_rng(seed); W = len(sA)
    idx = rng.integers(0, W, (B, W))
    pa = np.exp(sA[idx].sum(1) / nA[idx].sum(1)); pb = np.exp(sB[idx].sum(1) / nB[idx].sum(1))
    d = pa - pb
    point = math.exp(sA.sum() / nA.sum()) - math.exp(sB.sum() / nB.sum())
    return dict(diff=point, lo=float(np.percentile(d, 2.5)), hi=float(np.percentile(d, 97.5)))


def boot_acc_diff(cA, cB, B=BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n = len(cA)
    idx = rng.integers(0, n, (B, n))
    d = cA[idx].mean(1) - cB[idx].mean(1)
    return dict(diff=float(cA.mean() - cB.mean()), lo=float(np.percentile(d, 2.5)), hi=float(np.percentile(d, 97.5)))


def hs_correct(r, norm=True):
    ll, nc, lab = np.array(r["ll"]), np.array(r["nchar"]), np.array(r["label"])
    return ((ll / nc).argmax(1) == lab).astype(float) if norm else (ll.argmax(1) == lab).astype(float)

## Load the model (once) and record the environment

In [ ]:
# ============================================================
# Load the unchanged FP16 model ONCE; record the environment
# ============================================================
seed_all()
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Resume support: if a previous run of THIS notebook is attached as an input, continue from its outputs
PREV = sorted(glob.glob("/kaggle/input/*/t1_outputs")) + sorted(glob.glob("/kaggle/input/*/*/t1_outputs"))
if PREV and not (OUT_DIR / "results").exists():
    shutil.copytree(PREV[0], OUT_DIR, dirs_exist_ok=True)
    print("Resuming from previous outputs:", PREV[0], "->", sorted(p.name for p in (OUT_DIR / "results").glob("*.json")))

import transformers, huggingface_hub
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import HfApi

REVISION = HfApi().model_info(MODEL_ID).sha              # exact checkpoint revision
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=DTYPE,
                                             attn_implementation="sdpa").to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)
ENV = dict(model=MODEL_ID, revision=REVISION, tokenizer=tok.__class__.__name__, torch=torch.__version__,
           transformers=transformers.__version__, huggingface_hub=huggingface_hub.__version__,
           cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
           attn_implementation=model.config._attn_implementation, dtype=str(DTYPE), seed=SEED,
           python=platform.python_version(), tie_word_embeddings=model.config.tie_word_embeddings)
save_json(ENV, OUT_DIR / "environment.json")
print(json.dumps(ENV, indent=1))
check("transformers pinned version", "4.57.1", transformers.__version__)
check("huggingface_hub pinned version", "0.36.2", huggingface_hub.__version__)
check("base (not instruct) checkpoint", MODEL_ID, model.config._name_or_path)
check("attention backend", "sdpa", model.config._attn_implementation)

## Fixed data selections
Calibration = 128 × 512-token windows from C4-train (one random window per long document); development = 32 more from *different*
documents (used only for choosing P3/Q3 hyper-parameters). Held-out: 50,000 scored tokens of WikiText-2 test and of C4 validation,
and 2,000 HellaSwag validation IDs (the first 512 are the paired diagnostic subset).

In [ ]:
# ============================================================
# Fixed data selections (calibration, development, WikiText-2, C4 validation, HellaSwag)
# Every ID / start offset / token window is SAVED, so reruns and the report use identical data.
# ============================================================
from datasets import load_dataset
MAN_PATH = OUT_DIR / "manifests" / "manifests.json"


def tok_ids(text):
    return tok(text, add_special_tokens=False).input_ids


def build_c4_calib_dev():
    ds = load_dataset("allenai/c4", data_files={"train": "en/c4-train.00000-of-01024.json.gz"}, split="train")
    rng = np.random.default_rng(SEED)
    calib, dev = [], []
    for di in rng.permutation(len(ds)):
        ids = tok_ids(ds[int(di)]["text"])
        if len(ids) <= SEQ_LEN:
            continue
        st = int(rng.integers(0, len(ids) - SEQ_LEN + 1))
        rec = dict(doc_index=int(di), url=ds[int(di)]["url"], start=st, input_ids=ids[st:st + SEQ_LEN])
        (calib if len(calib) < CALIB_N else dev).append(rec)
        if len(dev) == DEV_N:
            break
    return calib, dev


def windows_from_docs(docs, target=EVAL_SCORED_TOKENS):
    """Each document tokenized separately; non-overlapping <=512-token windows; the last window
    is trimmed so that exactly `target` tokens are scored (first token of each window unscored)."""
    wins, total = [], 0
    for doc_id, text in docs:
        ids = tok_ids(text)
        for st in range(0, len(ids), WIN):
            w = ids[st:st + WIN]
            if len(w) < 2:
                continue
            need = target - total
            if len(w) - 1 > need:
                w = w[:need + 1]
            wins.append(dict(doc=doc_id, start=st, input_ids=w)); total += len(w) - 1
            if total == target:
                return wins
    raise RuntimeError("ran out of documents")


def wikitext_docs():
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    docs, cur = [], []
    for line in ds["text"]:
        if re.match(r"^ = [^=].* = \n$", line) and cur:      # a top-level " = Title = " starts a new article
            docs.append("".join(cur)); cur = []
        cur.append(line)
    if cur:
        docs.append("".join(cur))
    return [(i, d) for i, d in enumerate(docs) if d.strip()]


def c4_val_docs():
    ds = load_dataset("allenai/c4", data_files={"validation": "en/c4-validation.00000-of-00008.json.gz"}, split="validation")
    for di in np.random.default_rng(SEED).permutation(len(ds)):
        yield int(di), ds[int(di)]["text"]


def hellaswag_rows():
    try:
        ds = load_dataset("Rowan/hellaswag", split="validation")
        rows, src = [dict(r) for r in ds], "HF Rowan/hellaswag (validation)"
    except Exception as e:
        print("HF hellaswag failed, using the original release file:", repr(e)[:120])
        import urllib.request
        url = "https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_val.jsonl"
        rows, src = [json.loads(l) for l in urllib.request.urlopen(url).read().decode().splitlines() if l.strip()], url
    for r in rows:
        r["ind"], r["label"] = int(r["ind"]), int(r["label"])
    return rows, src


HS_ROWS, HS_SOURCE = hellaswag_rows()
if MAN_PATH.exists():
    MAN = load_json(MAN_PATH); print("Loaded saved manifests from", MAN_PATH)
else:
    t0 = time.perf_counter()
    calib, dev = build_c4_calib_dev()
    wt = windows_from_docs(wikitext_docs())
    c4v = windows_from_docs(c4_val_docs())
    hs_pick = np.random.default_rng(SEED).choice(len(HS_ROWS), HS_N, replace=False)
    MAN = dict(seed=SEED, tokenizer=MODEL_ID, revision=REVISION,
               calib=calib, dev=dev, wikitext=wt, c4_val=c4v,
               hellaswag_ids=[HS_ROWS[int(i)]["ind"] for i in hs_pick], hellaswag_source=HS_SOURCE,
               hellaswag_format="lm-eval-harness `hellaswag`: query = preprocess(activity_label + ': ' + ctx_a + ' ' + "
                                "ctx_b.capitalize()); continuation = ' ' + preprocess(ending); acc_norm divides the "
                                "summed continuation log-likelihood by len(ending) in characters",
               sources=dict(calib_dev="allenai/c4 en/c4-train.00000-of-01024.json.gz",
                            wikitext="Salesforce/wikitext wikitext-2-raw-v1 test",
                            c4_val="allenai/c4 en/c4-validation.00000-of-00008.json.gz"),
               build_seconds=time.perf_counter() - t0)
    save_json(MAN, MAN_PATH); print(f"Built and saved manifests in {MAN['build_seconds']:.0f}s")

CALIB_SEQS = [r["input_ids"] for r in MAN["calib"]]
DEV_SEQS = [r["input_ids"] for r in MAN["dev"]]
WT_WINDOWS = [r["input_ids"] for r in MAN["wikitext"]]
C4_WINDOWS = [r["input_ids"] for r in MAN["c4_val"]]
_by_ind = {r["ind"]: r for r in HS_ROWS}
HS_ENC = hs_encode(tok, [_by_ind[i] for i in MAN["hellaswag_ids"]])
PROMPT_IDS = [t for w in C4_WINDOWS for t in w][:2048]          # fixed timing prompt

check("calibration manifest: 128 sequences, every one 512 tokens", (CALIB_N, [SEQ_LEN]),
      (len(CALIB_SEQS), sorted(set(map(len, CALIB_SEQS)))))
check("development manifest: 32 sequences, every one 512 tokens", (DEV_N, [SEQ_LEN]),
      (len(DEV_SEQS), sorted(set(map(len, DEV_SEQS)))))
check("calibration and development documents are disjoint", 0,
      len({r["doc_index"] for r in MAN["calib"]} & {r["doc_index"] for r in MAN["dev"]}))
check("WikiText-2 scored tokens", EVAL_SCORED_TOKENS, sum(len(w) - 1 for w in WT_WINDOWS))
check("C4-validation scored tokens", EVAL_SCORED_TOKENS, sum(len(w) - 1 for w in C4_WINDOWS))
check("no window longer than 512", True, max(map(len, WT_WINDOWS + C4_WINDOWS)) <= WIN)
check("HellaSwag manifest: 2000 distinct validation IDs", HS_N, len(set(MAN["hellaswag_ids"])))
print("HellaSwag example:", HS_ENC[0]["query"][:150], "| endings:", [c[:40] for c in HS_ENC[0]["choices"]])

## Step 1 — Baseline and measurement protocol
Checks that the eligible set and the tied embedding are enumerated correctly, that the block runner equals a normal forward pass,
then evaluates the unchanged model with exactly the code every later condition uses.

In [ ]:
# ============================================================
# STEP 1 — the unchanged FP16 baseline and the measurement protocol
# ============================================================
RES_DIR, PEX_DIR = OUT_DIR / "results", OUT_DIR / "per_example"
for d in (RES_DIR, PEX_DIR):
    d.mkdir(parents=True, exist_ok=True)

ACC = param_accounting(model)
SHAPES = elig_shapes(model)
N_LAYERS = ACC["n_layers"]
print(json.dumps(ACC, indent=1))
check("eligible matrices = 7 per block x n_layers", 7 * N_LAYERS, ACC["n_eligible_matrices"])
check("lm_head shares storage with embed_tokens (tied => counted once)", True, ACC["lm_head_tied_to_embeddings"])
emb = model.model.embed_tokens.weight.numel()
check("embeddings appear once in the unique-parameter total",
      "total == eligible + non-eligible, embeddings counted once",
      f"total={ACC['n_total_unique']:,}  eligible={ACC['n_eligible']:,}  non-eligible={ACC['n_noneligible']:,}  (embedding {emb:,})",
      ok=ACC["n_total_unique"] == ACC["n_eligible"] + ACC["n_noneligible"])
check("every eligible matrix has in_features divisible by the group size 128 (and by 4 for 2:4)",
      True, all(s[1] % GROUP_SIZE == 0 and s[1] % 4 == 0 for s in SHAPES.values()))
excluded = [n for n, p in model.named_parameters()
            if not any(n.endswith(f"{p_}.weight") for p_ in PROJS)]
print("Excluded (never compressed) parameter tensors, e.g.:", excluded[:3], "...", excluded[-3:])

with torch.no_grad():                       # block-by-block runner == ordinary forward?
    _H = embed_batches(model, CALIB_SEQS[:2], bs=2)
    for _layer in model.model.layers:
        _H[0][0] = run_layer(_layer, *_H[0])
    _blockwise = model.model.norm(_H[0][0]).float()
    _full = model.model(input_ids=torch.tensor(CALIB_SEQS[:2], device=model.device), use_cache=False).last_hidden_state.float()
    _rel = float((_blockwise - _full).norm() / _full.norm())
check("block-by-block runner reproduces the full forward pass", "relative difference < 1e-2 (FP16)", f"{_rel:.2e}", ok=_rel < 1e-2)
del _H, _blockwise, _full

SNAP = snapshot(model)                      # unchanged FP16 checkpoint on CPU
BLOCK_PARAMS = [sum(SHAPES[f"{l}.{p}"][0] * SHAPES[f"{l}.{p}"][1] for p in PROJS) for l in range(N_LAYERS)]
ERR_SEQS = CALIB_SEQS[:ERR_N]               # fixed subset of the SAME calibration IDs
PROBE_SEQS = CALIB_SEQS[:PROBE_N]
DENSE_REF = dict(nll=calib_nll(model, ERR_SEQS), hidden=final_hidden(model, ERR_SEQS))
print(f"Unchanged model: calibration NLL on {ERR_N} calibration sequences = {DENSE_REF['nll']:.4f}")


def pname(n):
    li, p = n.split(".")
    return f"model.layers.{li}.{'self_attn' if p in ATTN_PROJS else 'mlp'}.{p}.weight"

In [ ]:
# ============================================================
# ONE FUNCTION RUNS EVERY CONDITION THE SAME WAY (identical data, scoring, timing)
# ============================================================
MEASURE_LATENCY = True


def evaluate_and_save(name, family, meta, hs_mode, storage, comp_time, lerr, extra, latency=True):
    calib = calib_metrics(model, ERR_SEQS, DENSE_REF)
    calib["dev_nll"] = calib_nll(model, DEV_SEQS)
    wt, wt_s, wt_n = eval_ppl(model, WT_WINDOWS)
    c4, c4_s, c4_n = eval_ppl(model, C4_WINDOWS)
    enc = HS_ENC if hs_mode == "full" else HS_ENC[:HS_SUB]
    hs, hs_pe = eval_hellaswag(model, enc)
    lat = measure_latency(model, PROMPT_IDS) if (latency and MEASURE_LATENCY) else None
    rec = dict(name=name, family=family, meta=meta, hs_mode=hs_mode, calib=calib, wikitext=wt, c4=c4,
               hellaswag=hs, storage=storage, latency=lat, compression_time_s=comp_time,
               local_errors=lerr, extra=extra, finished=time.strftime("%Y-%m-%d %H:%M:%S"))
    save_json(rec, RES_DIR / f"{name}.json")
    np.savez_compressed(PEX_DIR / f"{name}.npz", wt_sums=wt_s, wt_counts=wt_n, c4_sums=c4_s, c4_counts=c4_n,
                        hs_ll=hs_pe["ll"], hs_nchar=hs_pe["nchar"], hs_label=hs_pe["label"], hs_ids=hs_pe["ids"])
    print(f"   -> {name}: WikiText PPL {wt['ppl']:.3f} | C4 PPL {c4['ppl']:.3f} | HellaSwag({hs_mode}, n={hs['n']}) "
          f"acc {hs['acc']:.4f} acc_norm {hs['acc_norm']:.4f} | eligible MB {mb(storage['eligible_bytes']):.1f} "
          f"| calib dNLL {calib['calib_delta_nll']:+.4f}")
    return rec


def run_condition(name, family, meta, build, hs_mode="subset", latency=True):
    """build(): compress `model` IN PLACE from the unchanged weights and return
    dict(storage=..., extra=..., weff=optional fn(name, W_unchanged)->effective W)."""
    path = RES_DIR / f"{name}.json"
    if path.exists():
        print(f"[skip] {name} already finished (loaded from {path})")
        return load_json(path)
    print(f"\n=== {name} ===")
    try:
        restore_(model, SNAP); seed_all(); free()
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        sync(); t0 = time.perf_counter()
        info = build()
        sync(); comp_time = time.perf_counter() - t0
        comp_peak = mb(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else None
        # layer-output error is measured on UNCHANGED-model calibration activations
        stash = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if ".layers." in n}
        restore_(model, SNAP)
        weff = info.get("weff") or (lambda n, W: stash[pname(n)])
        lerr = local_errors(model, weff, ERR_SEQS)
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in stash: p.copy_(stash[n].to(p.device))
        del stash; free()
        extra = info.get("extra", {}); extra["compression_peak_alloc_MB"] = comp_peak
        return evaluate_and_save(name, family, meta, hs_mode, info["storage"], comp_time, lerr, extra, latency)
    except Exception as e:
        traceback.print_exc()
        rec = dict(name=name, family=family, meta=meta, failed=True, error=repr(e))
        save_json(rec, RES_DIR / f"{name}.json")
        return rec
    finally:
        restore_(model, SNAP); free()

In [ ]:
base = run_condition("FP16_baseline", "baseline", dict(),
                     lambda: dict(storage=storage_report(ACC, SHAPES, "dense", nnz=nonzero_counts(model))),
                     hs_mode="full")
check("baseline perplexity is finite on both suites", True,
      math.isfinite(base["wikitext"]["ppl"]) and math.isfinite(base["c4"]["ppl"]))
check("scored tokens per suite", EVAL_SCORED_TOKENS, (base["wikitext"]["N"], base["c4"]["N"]),
      ok=base["wikitext"]["N"] == EVAL_SCORED_TOKENS == base["c4"]["N"])
check("baseline layer-output error is zero (W_eff == W)", 0.0, max(base["local_errors"].values()),
      ok=max(base["local_errors"].values()) < 1e-10)
check("HellaSwag accuracy above 25% chance (zero-shot base 1.5B is typically ~0.5 acc / ~0.65 acc_norm)",
      "> 0.25", f"acc={base['hellaswag']['acc']:.4f} acc_norm={base['hellaswag']['acc_norm']:.4f}",
      ok=base["hellaswag"]["acc"] > 0.25)
print(json.dumps(base["latency"], indent=1)[:1500] if base.get("latency") else "")

## Step 2 — P1 magnitude pruning
Score = |W|. In every output row the lowest 25/50/75% are zeroed. Look for: achieved sparsity exactly equal to the target; how fast
perplexity rises from 50% to 75%.

In [ ]:
# ============================================================
# STEP 2 — P1 magnitude pruning, per output row, s in {0.25, 0.50, 0.75}
# ============================================================
def build_p1(S):
    def build():
        magnitude_prune_(model, uniform_k(S))
        return dict(storage=storage_report(ACC, SHAPES, "sparse", nnz=nonzero_counts(model)),
                    extra=dict(target_sparsity=S))
    return build


for S in (0.25, 0.50, 0.75):
    r = run_condition(f"P1_mag_s{int(S*100)}", "P1", dict(sparsity=S), build_p1(S),
                      hs_mode="full" if S == 0.5 else "subset")
    if not r.get("failed"):
        check(f"P1 s={S}: achieved eligible sparsity", S, round(r["storage"]["achieved_sparsity"], 6),
              ok=abs(r["storage"]["achieved_sparsity"] - S) < 1e-6)

## Step 3 — P2 Wanda
Score = |W_ij| · ‖X_j‖₂ (the input feature's norm over all 128×512 calibration tokens), same per-row masking. Blocks are processed
in order and each block's inputs come from the already-pruned earlier blocks (as in the official Wanda code). The printed *mask
overlap* = share of Wanda-pruned weights that magnitude would also have pruned — the gap is exactly where activations changed the decision.

In [ ]:
# ============================================================
# STEP 3 — P2 Wanda (|W_ij| * ||X_j||_2), same per-row masking, same sparsities
# ============================================================
def build_p2(S):
    def build():
        rec = {}
        wanda_prune_sequential_(model, CALIB_SEQS, uniform_k(S), record=rec)
        return dict(storage=storage_report(ACC, SHAPES, "sparse", nnz=nonzero_counts(model)),
                    extra=dict(target_sparsity=S, score_stats=rec))
    return build


for S in (0.25, 0.50, 0.75):
    r = run_condition(f"P2_wanda_s{int(S*100)}", "P2", dict(sparsity=S), build_p2(S),
                      hs_mode="full" if S == 0.5 else "subset")
    if not r.get("failed"):
        check(f"P2 s={S}: achieved eligible sparsity", S, round(r["storage"]["achieved_sparsity"], 6),
              ok=abs(r["storage"]["achieved_sparsity"] - S) < 1e-6)
        ov = np.mean([v["mask_overlap_with_magnitude"] for v in r["extra"]["score_stats"].values()])
        print(f"   mean fraction of Wanda-pruned weights that magnitude would ALSO prune: {ov:.3f}")

## Step 4 — P3 non-uniform Wanda
Same Wanda ranking *inside* a block; only the per-block sparsity changes. Two sensitivity measures:
**(A) OWL outlier ratio** — share of a block's Wanda scores above 5× the block mean (outlier-rich blocks are pruned less);
**(B) isolated-removal probe** — how much the calibration NLL rises when *only* block ℓ is pruned at the global target.
Allocation rule: s_ℓ = S − λ·u_ℓ (u = normalized sensitivity in [−1, 1]), then one common shift so the total number of zeros
equals P1/P2 exactly, with every s_ℓ in [0.10, 0.85]. The probe takes ~10 min.

In [ ]:
# ============================================================
# STEP 4a — P3 sensitivity measures (calibration data only)
#   (A) OWL layerwise outlier distribution of Wanda scores      (cheap, from one dense pass)
#   (B) isolated-removal probe: dNLL when ONLY block l is Wanda-pruned at the global target
# ============================================================
SENS_PATH = OUT_DIR / "p3_sensitivity.json"
if SENS_PATH.exists():
    P3_SENS = load_json(SENS_PATH)
else:
    restore_(model, SNAP)
    t0 = time.perf_counter()
    DENSE_STATS = dense_input_stats(model, CALIB_SEQS)
    lod = owl_lod(model, DENSE_STATS)
    t_owl = time.perf_counter() - t0
    base_probe = calib_nll(model, PROBE_SEQS)
    probe, t0 = {}, time.perf_counter()
    for S in (0.5, 0.75):
        probe[str(S)] = []
        for li in range(N_LAYERS):
            wanda_prune_block_dense_(model, li, S, DENSE_STATS)
            probe[str(S)].append(calib_nll(model, PROBE_SEQS) - base_probe)
            restore_(model, SNAP, prefix=f"model.layers.{li}.")
        print(f"   probe S={S}: most sensitive blocks {np.argsort(probe[str(S)])[::-1][:5].tolist()}")
    P3_SENS = dict(owl_lod=lod.tolist(), owl_M=OWL_M, probe_dnll=probe, probe_base_nll=base_probe,
                   probe_n_seqs=PROBE_N, time_owl_s=t_owl, time_probe_s=time.perf_counter() - t0)
    save_json(P3_SENS, SENS_PATH)
    del DENSE_STATS; free()
check("model restored after probes (calibration NLL unchanged)", round(DENSE_REF["nll"], 4),
      round(calib_nll(model, ERR_SEQS), 4), ok=abs(calib_nll(model, ERR_SEQS) - DENSE_REF["nll"]) < 1e-3)
print("OWL LOD per block:", np.round(P3_SENS["owl_lod"], 4).tolist())

Candidates (2 measures × λ ∈ {0.04, 0.08}) are compared **only on the development NLL**; the choice is written to
`p3_selection.json` *before* any held-out evaluation.

In [ ]:
# ============================================================
# STEP 4b — P3 candidate allocations, chosen ONLY on the 32-sequence C4-train DEV manifest
# ============================================================
SEL_PATH = OUT_DIR / "p3_selection.json"


def p3_target_zeros(S):
    return sum(d_out * int(round(S * d_in)) for d_out, d_in in SHAPES.values())


def p3_candidates(S):
    cands = []
    for measure, r, mode in [("owl", P3_SENS["owl_lod"], "minmax"), ("probe", P3_SENS["probe_dnll"][str(S)], "rank")]:
        for lam in P3_LAMBDAS:
            s_blocks = allocate_sparsity(r, S, lam, mode, BLOCK_PARAMS)
            cands.append(dict(id=f"{measure}_lam{lam}", measure=measure, lam=lam, norm=mode, s_blocks=s_blocks.tolist()))
    return cands


def k_fn_from(K):
    return lambda li, p, dout, din: K[f"{li}.{p}"]


if SEL_PATH.exists():
    P3_SEL = load_json(SEL_PATH)
else:
    P3_SEL = {}
    for S in (0.5, 0.75):
        tz = p3_target_zeros(S)
        rows = []
        for c in p3_candidates(S):
            K = k_rows_from_block_sparsity(c["s_blocks"], SHAPES, tz)
            restore_(model, SNAP); t0 = time.perf_counter()
            wanda_prune_sequential_(model, CALIB_SEQS, k_fn_from(K))
            zeros = sum(int((m.weight == 0).sum()) for m in eligible_linears(model).values())
            c.update(dev_nll=calib_nll(model, DEV_SEQS), zeros=zeros, build_s=time.perf_counter() - t0,
                     realized_block_sparsity=[sum(int(K[f"{l}.{p}"].sum()) for p in PROJS) / BLOCK_PARAMS[l]
                                              for l in range(N_LAYERS)])
            rows.append(c)
            print(f"   S={S} {c['id']:>16}: dev NLL {c['dev_nll']:.4f} | zeros {zeros:,} (target {tz:,})")
        restore_(model, SNAP)
        best = min(rows, key=lambda c: c["dev_nll"])
        P3_SEL[str(S)] = dict(target_zeros=tz, candidates=rows, selected=best["id"],
                              rule="lowest DEV NLL (C4-train dev manifest); held-out sets never consulted",
                              frozen_at=time.strftime("%Y-%m-%d %H:%M:%S"))
    save_json(P3_SEL, SEL_PATH)          # selection is frozen on disk BEFORE any held-out evaluation
for S in ("0.5", "0.75"):
    sel = P3_SEL[S]
    print(f"S={S}: selected {sel['selected']}")
    for c in sel["candidates"]:
        check(f"P3 S={S} {c['id']}: zeroed eligible weights equal the P1/P2 budget", sel["target_zeros"], c["zeros"])
        lo, hi = min(c["s_blocks"]), max(c["s_blocks"])
        check(f"P3 S={S} {c['id']}: every block sparsity inside [0.10, 0.85]", "[0.10, 0.85]", f"[{lo:.3f}, {hi:.3f}]",
              ok=lo >= SPARSITY_BOUNDS[0] - 1e-3 and hi <= SPARSITY_BOUNDS[1] + 1e-3)

The selected allocation becomes `P3_nonuni_s50/75`. The unselected candidates are also evaluated (subset HellaSwag) and kept as
`P3cand_*` — the assignment asks you to retain unsuccessful designs and test whether dev improvements carry over.

In [ ]:
# ============================================================
# STEP 4c — evaluate the frozen P3 selections (and keep the unselected designs as diagnostics)
# ============================================================
def build_p3(S, cand):
    def build():
        K = k_rows_from_block_sparsity(cand["s_blocks"], SHAPES, P3_SEL[str(S)]["target_zeros"])
        rec = {}
        wanda_prune_sequential_(model, CALIB_SEQS, k_fn_from(K), record=rec)
        return dict(storage=storage_report(ACC, SHAPES, "sparse", nnz=nonzero_counts(model)),
                    extra=dict(target_sparsity=S, candidate=cand, score_stats=rec))
    return build


for S in (0.5, 0.75):
    sel = P3_SEL[str(S)]
    for c in sel["candidates"]:
        chosen = c["id"] == sel["selected"]
        name = f"P3_nonuni_s{int(S*100)}" if chosen else f"P3cand_s{int(S*100)}_{c['id']}"
        r = run_condition(name, "P3" if chosen else "P3_unselected", dict(sparsity=S, candidate=c["id"], selected=chosen),
                          build_p3(S, c), hs_mode="full" if (chosen and S == 0.5) else "subset", latency=chosen)
        if chosen and not r.get("failed"):
            p2 = load_json(RES_DIR / f"P2_wanda_s{int(S*100)}.json")
            check(f"P3 S={S}: nonzeros equal P2 at matched sparsity", p2["storage"]["eligible_nonzeros"],
                  r["storage"]["eligible_nonzeros"])

## Step 8 — 2:4 structured sparsity (quality/representation study only — no sparse-kernel speedup is claimed)

In [ ]:
# ============================================================
# STEP 8 — structured 2:4 diagnostic vs unstructured 50% Wanda
# ============================================================
def build_24():
    wanda_prune_sequential_(model, CALIB_SEQS, pattern="2:4")
    nnz = nonzero_counts(model)
    st = storage_report(ACC, SHAPES, "sparse", nnz=nnz)
    n = ACC["n_eligible"]
    st["eligible_2of4_compressed_bytes"] = n // 2 * 2 + math.ceil(n // 2 * 2 / 8)   # FP16 values + 2-bit index per kept weight
    return dict(storage=st, extra=dict(pattern="2:4"))


r24 = run_condition("P2_wanda_2of4", "P2_24", dict(sparsity=0.5, pattern="2:4"), build_24)
if not r24.get("failed"):
    check("2:4 achieved sparsity equals 50%", 0.5, r24["storage"]["achieved_sparsity"])
    with torch.no_grad():   # verify the pattern on one rebuilt matrix
        restore_(model, SNAP)
        W = get_proj(model.model.layers[0], "q_proj").weight
        m = mask_24(W.abs().float())
        check("mask_24 keeps exactly 2 of every 4 consecutive input weights (function test on block-0 q_proj)", 2,
              int((~m).reshape(W.shape[0], -1, 4).sum(-1).unique().item()))

## Steps 5–6 — Quantization
**AWQ search** (one dense pass): for every block and input group, scales s = (mean|x|)^α normalized, α from {0, 0.1, …, 1.0},
choosing the α with the lowest calibration output error of Q(W·s)/s. Done for 2/3/4/8 bits at once (Q3 reuses them).
α = 0 means s = 1, i.e. plain Q1 — so AWQ can never be worse than Q1 *on its own calibration objective* (checked below).

In [ ]:
# ============================================================
# STEP 6a — AWQ statistics + scale search (calibration activations only), all bit widths at once
# ============================================================
AWQ_PATH = OUT_DIR / "awq_search.pt"
if AWQ_PATH.exists():
    AWQ = torch.load(AWQ_PATH, weights_only=False)
else:
    restore_(model, SNAP)
    AWQ = {b: {} for b in Q3_BITS}
    t0 = time.perf_counter()

    def cb(li, layer, samples, st):
        res = awq_search_layer(layer, {g: samples[g].to(layer.input_layernorm.weight.device) for g in GROUPS},
                               st, Q3_BITS)
        for b in Q3_BITS:
            AWQ[b][li] = res[b]
        if li % 7 == 0:
            print(f"   layer {li}: chosen alpha at 4-bit " +
                  ", ".join(f"{g}={res[4][g]['alpha']}" for g in GROUPS) +
                  f" | rel err gu 4-bit: alpha0 {res[4]['gu']['rel_err_alpha0']:.2e} -> best {res[4]['gu']['rel_err_best']:.2e}")
    dense_input_stats(model, CALIB_SEQS, n_sample=AWQ_TOKENS, layer_callback=cb)
    AWQ["meta"] = dict(time_s=time.perf_counter() - t0, grid=AWQ_GRID, tokens_per_layer=AWQ_TOKENS,
                       stat="mean |x| over all calibration tokens", objective="calibration output MSE")
    torch.save(AWQ, AWQ_PATH)
print(f"AWQ search time: {AWQ['meta']['time_s']:.1f}s")
worse = sum(AWQ[b][l][g]["rel_err_best"] > AWQ[b][l][g]["rel_err_alpha0"] + 1e-12
            for b in Q3_BITS for l in range(N_LAYERS) for g in GROUPS)
check("AWQ search never worse than plain quantization on its calibration objective (alpha=0 is in the grid)", 0, worse)

**Fold check:** apply the scales *without rounding*; the model must compute the same function (only FP16 rounding noise).

In [ ]:
# ============================================================
# STEP 6b — verify that full-precision rescaling (fold, NO rounding) preserves the function
# ============================================================
restore_(model, SNAP)
FOLD_CHECK = {}
with torch.no_grad():
    ids = torch.tensor(ERR_SEQS[:4], device=model.device)
    ref_logits = model(input_ids=ids, use_cache=False).logits.float()
    for b in Q3_BITS:                       # the scales searched for each bit width
        for li, layer in enumerate(model.model.layers):
            apply_awq_block_(layer, {g: AWQ[b][li][g]["scale"] for g in GROUPS}, b, quantize=False)
        new_logits = model(input_ids=ids, use_cache=False).logits.float()
        kl = float(F.kl_div(F.log_softmax(new_logits, -1), F.log_softmax(ref_logits, -1), log_target=True,
                            reduction="none").sum(-1).mean())
        top1 = float((new_logits.argmax(-1) == ref_logits.argmax(-1)).float().mean())
        maxdiff = float((new_logits - ref_logits).abs().max())
        FOLD_CHECK[f"scales_for_{b}bit"] = dict(mean_token_KL=kl, top1_agreement=top1, max_abs_logit_diff=maxdiff)
        check(f"{b}-bit AWQ scales folded WITHOUT rounding reproduce the unchanged model (FP16 rounding only)",
              "KL < 1e-3, top-1 agreement > 0.99", f"KL={kl:.2e}, top1={top1:.4f}, max|dlogit|={maxdiff:.3f}",
              ok=kl < 1e-3 and top1 > 0.99)
        restore_(model, SNAP)
        del new_logits
save_json(FOLD_CHECK, OUT_DIR / "awq_fold_check.json")

**Q1 vs Q2** at 8/4/3 bits. Q2's only extra bytes are the 28 × 1536 FP16 scales for `o_proj`, which cannot be folded into
`v_proj` because of grouped-query attention (v has 256 outputs, o has 1536 inputs).

In [ ]:
# ============================================================
# STEPS 5 & 6c — Q1 uniform RTN quantization and Q2 AWQ-style, b in {8, 4, 3}
# ============================================================
def bits_map(bits_by_layer):
    return {n: bits_by_layer[int(n.split(".")[0])] for n in SHAPES}


def build_q1(b):
    def build():
        q1_quantize_(model, lambda li: b)
        return dict(storage=storage_report(ACC, SHAPES, "quant", bits=bits_map([b] * N_LAYERS)),
                    extra=dict(bits=b), weff=lambda n, W: fake_quant(W, b)[0])
    return build


def build_awq(bits_by_layer):
    def build():
        for li, layer in enumerate(model.model.layers):
            apply_awq_block_(layer, {g: AWQ[bits_by_layer[li]][li][g]["scale"] for g in GROUPS}, bits_by_layer[li])

        def weff(n, W):
            li, p = int(n.split(".")[0]), n.split(".")[1]
            s = AWQ[bits_by_layer[li]][li][GROUP_OF[p]]["scale"].to(W.device).float()
            return awq_effective_weight(W, s, bits_by_layer[li])
        return dict(storage=storage_report(ACC, SHAPES, "quant", bits=bits_map(bits_by_layer),
                                           awq_o_scale_bytes=awq_o_scale_bytes(model)),
                    extra=dict(bits_by_layer=list(bits_by_layer),
                               alphas={li: {g: AWQ[bits_by_layer[li]][li][g]["alpha"] for g in GROUPS} for li in range(N_LAYERS)}),
                    weff=weff)
    return build


# packing check on a real matrix: the byte formula equals an actual packed buffer
with torch.no_grad():
    W0 = get_proj(model.model.layers[0], "q_proj").weight
    for b in (8, 4, 3, 2):
        _, q, sc, z = fake_quant(W0, b)
        packed = pack_bits(q.cpu().numpy(), b)
        rt = unpack_bits(packed, b, q.numel())
        actual = packed.nbytes + sc.numel() * 2 + z.numel()
        check(f"{b}-bit pack/unpack round-trip exact and bytes == formula", quant_tensor_bytes(*W0.shape, b),
              actual, ok=(np.array_equal(rt, q.cpu().numpy().reshape(-1)) and actual == quant_tensor_bytes(*W0.shape, b)))
        check(f"{b}-bit: at most 2^b distinct codes", f"<= {2**b}", int(q.unique().numel()), ok=q.unique().numel() <= 2 ** b)

for b in (8, 4, 3):
    run_condition(f"Q1_rtn_int{b}", "Q1", dict(bits=b), build_q1(b), hs_mode="full" if b in (4, 3) else "subset")
    r = run_condition(f"Q2_awq_int{b}", "Q2", dict(bits=b), build_awq([b] * N_LAYERS), hs_mode="full" if b in (4, 3) else "subset")
    if not r.get("failed"):
        q1 = load_json(RES_DIR / f"Q1_rtn_int{b}.json")
        print(f"   INT{b}: Q2 extra bytes vs Q1 (unfoldable o_proj scales) = {r['storage']['eligible_bytes'] - q1['storage']['eligible_bytes']:,}")
        e1, e2 = np.mean(list(q1["local_errors"].values())), np.mean(list(r["local_errors"].values()))
        print(f"   INT{b}: mean calibration layer-output rel. error Q1 {e1:.3e} -> Q2 {e2:.3e}")

## Step 7 — Q3 mixed precision
Sensitivity: calibration ΔNLL when *only* block ℓ is AWQ-quantized at 2/3/4/8 bits (~20 min), plus a free proxy (the AWQ search's
own relative error). Allocation: an exact dynamic program that minimises the summed sensitivity subject to total bits ≤ 28 × 4
(or 28 × 3). Because all blocks have identical shapes and metadata, that bit budget is exactly the Q2 packed-byte budget — the
packed bytes are still recomputed and checked.

In [ ]:
# ============================================================
# STEP 7a — Q3 sensitivity: dNLL when ONLY block l is AWQ-quantized at b in {2,3,4,8}
#            (+ a free proxy: the AWQ search's own relative output error)
# ============================================================
Q3S_PATH = OUT_DIR / "q3_sensitivity.json"
if Q3S_PATH.exists():
    Q3_SENS = load_json(Q3S_PATH)
else:
    restore_(model, SNAP)
    base_probe = calib_nll(model, PROBE_SEQS)
    probe, t0 = {str(b): [] for b in Q3_BITS}, time.perf_counter()
    for b in Q3_BITS:
        for li, layer in enumerate(model.model.layers):
            apply_awq_block_(layer, {g: AWQ[b][li][g]["scale"] for g in GROUPS}, b)
            probe[str(b)].append(calib_nll(model, PROBE_SEQS) - base_probe)
            restore_(model, SNAP, prefix=f"model.layers.{li}.")
        print(f"   {b}-bit probe: most sensitive blocks {np.argsort(probe[str(b)])[::-1][:5].tolist()}")
    proxy = {str(b): [float(sum(AWQ[b][l][g]["rel_err_best"] for g in GROUPS)) for l in range(N_LAYERS)] for b in Q3_BITS}
    Q3_SENS = dict(probe_dnll=probe, awq_proxy=proxy, base_nll=base_probe, time_probe_s=time.perf_counter() - t0)
    save_json(Q3_SENS, Q3S_PATH)

In [ ]:
# ============================================================
# STEP 7b — Q3 allocation: exact DP under each Q2 packed-byte budget, chosen on DEV only
# ============================================================
Q3SEL_PATH = OUT_DIR / "q3_selection.json"
assert len(set(tuple(SHAPES[f"{l}.{p}"] for p in PROJS) for l in range(N_LAYERS))) == 1, "blocks differ in shape"
if Q3SEL_PATH.exists():
    Q3_SEL = load_json(Q3SEL_PATH)
else:
    Q3_SEL = {}
    for bu in (4, 3):
        budget = load_json(RES_DIR / f"Q2_awq_int{bu}.json")["storage"]["total_model_bytes"]
        rows = []
        for measure in ("probe_dnll", "awq_proxy"):
            cost = [{b: Q3_SENS[measure][str(b)][l] for b in Q3_BITS} for l in range(N_LAYERS)]
            bits = dp_bits(cost, N_LAYERS * bu)
            st = storage_report(ACC, SHAPES, "quant", bits=bits_map(bits), awq_o_scale_bytes=awq_o_scale_bytes(model))
            restore_(model, SNAP)
            for li, layer in enumerate(model.model.layers):
                apply_awq_block_(layer, {g: AWQ[bits[li]][li][g]["scale"] for g in GROUPS}, bits[li])
            rows.append(dict(id=f"{measure}_dp", measure=measure, bits=bits, total_bytes=st["total_model_bytes"],
                             budget_bytes=budget, dev_nll=calib_nll(model, DEV_SEQS),
                             bit_histogram={b: bits.count(b) for b in Q3_BITS}))
            print(f"   INT{bu} budget {measure:>10}: bits {bits} | bytes {st['total_model_bytes']:,} <= {budget:,} "
                  f"| dev NLL {rows[-1]['dev_nll']:.4f}")
        restore_(model, SNAP)
        best = min(rows, key=lambda c: c["dev_nll"])
        Q3_SEL[str(bu)] = dict(candidates=rows, selected=best["id"], rule="lowest DEV NLL",
                               frozen_at=time.strftime("%Y-%m-%d %H:%M:%S"))
    save_json(Q3_SEL, Q3SEL_PATH)
for bu in ("4", "3"):
    for c in Q3_SEL[bu]["candidates"]:
        check(f"Q3 INT{bu}-budget {c['id']}: packed bytes do not exceed the Q2 uniform budget",
              f"<= {c['budget_bytes']:,}", f"{c['total_bytes']:,}", ok=c["total_bytes"] <= c["budget_bytes"])

In [ ]:
for bu in (4, 3):
    sel = Q3_SEL[str(bu)]
    for c in sel["candidates"]:
        chosen = c["id"] == sel["selected"]
        name = f"Q3_mixed_int{bu}budget" if chosen else f"Q3cand_int{bu}budget_{c['id']}"
        run_condition(name, "Q3" if chosen else "Q3_unselected", dict(budget_bits=bu, candidate=c["id"], selected=chosen),
                      build_awq(c["bits"]), hs_mode="full" if chosen else "subset", latency=chosen)

save_json(CHECKS, OUT_DIR / "checks.json")
print(f"\n{sum(c['ok'] for c in CHECKS)}/{len(CHECKS)} checks passed")

## Step 9 — Analysis (runs from the saved files; you can re-run just these cells later)

In [ ]:
# ============================================================
# STEP 9 — load every saved result (works on a fresh session from saved files alone)
# ============================================================
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = OUT_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
R = {Path(p).stem: load_json(p) for p in sorted(glob.glob(str(RES_DIR / "*.json")))}
PE = {k: dict(np.load(PEX_DIR / f"{k}.npz")) for k in R if (PEX_DIR / f"{k}.npz").exists()}
FAILED = [k for k, v in R.items() if v.get("failed")]
OK = [k for k in R if k not in FAILED]
print(f"{len(R)} conditions saved | failed: {FAILED if FAILED else 'none'}")

# fixed categorical slots (same entity -> same colour in every figure)
C = dict(uniform="#2a78d6", aware="#eb6834", adaptive="#1baf7a", struct="#eda100", base="#52514e", cand="#9a9893")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#dddcd8", "grid.linewidth": 0.6,
                     "axes.edgecolor": "#8a8984", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.spines.top": False, "axes.spines.right": False,
                     "lines.linewidth": 2, "lines.markersize": 7, "font.size": 10})
BASE = R["FP16_baseline"]; BASE_PE = PE["FP16_baseline"]
from matplotlib.ticker import MaxNLocator
INT_X = MaxNLocator(integer=True)

In [ ]:
# ============================================================
# Compact summary of every condition with PAIRED changes from the unchanged model
# ============================================================
def paired_hs_delta(k, norm=True):
    """acc(_norm) change vs baseline on exactly the same HellaSwag IDs."""
    ids = PE[k]["hs_ids"]; n = len(ids)
    rc = dict(ll=PE[k]["hs_ll"], nchar=PE[k]["hs_nchar"], label=PE[k]["hs_label"])
    rb = dict(ll=BASE_PE["hs_ll"][:n], nchar=BASE_PE["hs_nchar"][:n], label=BASE_PE["hs_label"][:n])
    assert np.array_equal(BASE_PE["hs_ids"][:n], ids)
    return float(hs_correct(rc, norm).mean() - hs_correct(rb, norm).mean())


rows = []
for k in OK:
    r = R[k]; st = r["storage"]; lat = r.get("latency") or {}
    rows.append(dict(
        condition=k, family=r["family"],
        sparsity=st.get("achieved_sparsity", 0.0), nominal_bits=st.get("nominal_bits_per_weight"),
        eff_bits_incl_meta=st["effective_bits_per_weight_incl_metadata"],
        eligible_MB=mb(st["eligible_bytes"]), total_MB=st["total_model_MB"],
        wt_ppl=r["wikitext"]["ppl"], d_wt_nll=r["wikitext"]["mean_nll"] - BASE["wikitext"]["mean_nll"],
        c4_ppl=r["c4"]["ppl"], d_c4_nll=r["c4"]["mean_nll"] - BASE["c4"]["mean_nll"],
        hs_n=r["hellaswag"]["n"], hs_acc=r["hellaswag"]["acc"], hs_acc_norm=r["hellaswag"]["acc_norm"],
        d_acc_norm_paired=paired_hs_delta(k), calib_dnll=r["calib"]["calib_delta_nll"], dev_nll=r["calib"]["dev_nll"],
        mean_layer_rel_err=float(np.mean(list(r["local_errors"].values()))),
        compress_s=r["compression_time_s"],
        prefill2048_ms=lat.get("prefill_2048", {}).get("mean"), decode_ms_tok=lat.get("decode_ms_per_token"),
        peak_MB_prefill2048=lat.get("peak_alloc_MB_prefill_2048")))
SUMMARY = pd.DataFrame(rows).set_index("condition")
SUMMARY.to_csv(OUT_DIR / "summary_all_conditions.csv")
main = SUMMARY[~SUMMARY.family.str.contains("unselected")]
pd.set_option("display.width", 250, "display.max_columns", 30)
print(main[["sparsity", "nominal_bits", "eligible_MB", "wt_ppl", "c4_ppl", "hs_n", "hs_acc", "hs_acc_norm",
            "d_acc_norm_paired", "calib_dnll"]].round(4).to_string())

In [ ]:
# ============================================================
# Paired 95% bootstrap intervals: P3 - P2 and Q3 - Q2 (and each vs baseline)
# ============================================================
def boot_pair(a, b):
    A, Bp = PE[a], PE[b]
    out = dict(pair=f"{a} - {b}")
    for suite in ("wt", "c4"):
        bt = boot_ppl_diff(A[f"{suite}_sums"], A[f"{suite}_counts"], Bp[f"{suite}_sums"], Bp[f"{suite}_counts"])
        out.update({f"{suite}_ppl_diff": bt["diff"], f"{suite}_lo": bt["lo"], f"{suite}_hi": bt["hi"]})
    n = min(len(A["hs_ids"]), len(Bp["hs_ids"]))
    assert np.array_equal(A["hs_ids"][:n], Bp["hs_ids"][:n])
    for norm, tag in ((True, "acc_norm"), (False, "acc")):
        ca = hs_correct(dict(ll=A["hs_ll"][:n], nchar=A["hs_nchar"][:n], label=A["hs_label"][:n]), norm)
        cb = hs_correct(dict(ll=Bp["hs_ll"][:n], nchar=Bp["hs_nchar"][:n], label=Bp["hs_label"][:n]), norm)
        bt = boot_acc_diff(ca, cb)
        out.update({f"hs_{tag}_diff_pp": 100 * bt["diff"], f"hs_{tag}_lo_pp": 100 * bt["lo"],
                    f"hs_{tag}_hi_pp": 100 * bt["hi"]})
    out["hs_n"] = n
    return out


pairs = [("P3_nonuni_s50", "P2_wanda_s50"), ("P3_nonuni_s75", "P2_wanda_s75"),
         ("Q3_mixed_int4budget", "Q2_awq_int4"), ("Q3_mixed_int3budget", "Q2_awq_int3"),
         ("P2_wanda_s50", "P1_mag_s50"), ("Q2_awq_int4", "Q1_rtn_int4"), ("Q2_awq_int3", "Q1_rtn_int3"),
         ("P2_wanda_2of4", "P2_wanda_s50")]
BOOTS = pd.DataFrame([boot_pair(a, b) for a, b in pairs if a in PE and b in PE])
BOOTS.to_csv(OUT_DIR / "paired_bootstrap.csv", index=False)
print(f"Paired bootstrap ({BOOT} resamples; PPL: whole windows resampled, NLL pooled, then exponentiated)")
print(BOOTS.round(3).to_string(index=False))

In [ ]:
# ============================================================
# Figure 1/2 — quality vs resources, pruning and quantization separated
# ============================================================
def get(k, f):
    return R[k][f]["ppl"] if k in OK else np.nan


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, suite, title in ((axes[0], "wikitext", "WikiText-2 test"), (axes[1], "c4", "C4 validation")):
    ax.axhline(BASE[suite]["ppl"], color=C["base"], ls="--", lw=1.2, label="FP16 unchanged")
    for fam, keys, col in (("P1 magnitude", ["P1_mag_s25", "P1_mag_s50", "P1_mag_s75"], C["uniform"]),
                           ("P2 Wanda", ["P2_wanda_s25", "P2_wanda_s50", "P2_wanda_s75"], C["aware"])):
        ax.plot([25, 50, 75], [get(k, suite) for k in keys], "o-", color=col, label=fam)
    ax.plot([50, 75], [get("P3_nonuni_s50", suite), get("P3_nonuni_s75", suite)], "s-", color=C["adaptive"],
            label="P3 non-uniform Wanda")
    ax.plot([50], [get("P2_wanda_2of4", suite)], "D", color=C["struct"], label="Wanda 2:4")
    ax.set_yscale("log"); ax.set_xlabel("eligible-weight sparsity (%)"); ax.set_ylabel("perplexity (log scale)")
    ax.set_title(f"Pruning — {title} ({BASE[suite]['N']:,} scored tokens)")
axes[0].legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig1_pruning_quality.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, suite, title in ((axes[0], "wikitext", "WikiText-2 test"), (axes[1], "c4", "C4 validation")):
    ax.plot([mb(BASE["storage"]["eligible_bytes"])], [BASE[suite]["ppl"]], "o", color=C["base"], label="FP16 unchanged")
    for fam, pre, col in (("Q1 RTN", "Q1_rtn_int", C["uniform"]), ("Q2 AWQ-style", "Q2_awq_int", C["aware"])):
        ks = [f"{pre}{b}" for b in (8, 4, 3) if f"{pre}{b}" in OK]
        ax.plot([mb(R[k]["storage"]["eligible_bytes"]) for k in ks], [R[k][suite]["ppl"] for k in ks], "o-", color=col, label=fam)
        for k in ks:
            ax.annotate(f"INT{k[-1]}", (mb(R[k]["storage"]["eligible_bytes"]), R[k][suite]["ppl"]),
                        textcoords="offset points", xytext=(4, 4), fontsize=8, color="#52514e")
    ks = [k for k in ("Q3_mixed_int4budget", "Q3_mixed_int3budget") if k in OK]
    ax.plot([mb(R[k]["storage"]["eligible_bytes"]) for k in ks], [R[k][suite]["ppl"] for k in ks], "s", color=C["adaptive"],
            label="Q3 mixed precision")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("packed eligible-weight storage incl. scales/zeros (MB, log)"); ax.set_ylabel("perplexity (log scale)")
    ax.set_title(f"Quantization — {title}")
axes[0].legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig2_quant_quality.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# Figure 3/4 — sensitivity measures and the allocations they produced
# ============================================================
P3_SENS = load_json(OUT_DIR / "p3_sensitivity.json"); P3_SEL = load_json(OUT_DIR / "p3_selection.json")
L = len(P3_SENS["owl_lod"]); x = np.arange(L)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].bar(x, P3_SENS["owl_lod"], color=C["aware"], width=0.8)
axes[0].set_title(f"OWL outlier ratio (Wanda score > {P3_SENS['owl_M']:g} x block mean)"); axes[0].set_xlabel("block")
for S, col in (("0.5", C["uniform"]), ("0.75", C["adaptive"])):
    axes[1].plot(x, P3_SENS["probe_dnll"][S], "o-", color=col, label=f"prune only this block at {float(S):.0%}")
axes[1].set_yscale("symlog", linthresh=1e-3); axes[1].set_title("Isolated-removal probe: calibration dNLL")
axes[1].set_xlabel("block"); axes[1].legend(frameon=False)
for S, col in (("0.5", C["uniform"]), ("0.75", C["adaptive"])):
    sel = next(c for c in P3_SEL[S]["candidates"] if c["id"] == P3_SEL[S]["selected"])
    axes[2].plot(x, sel["realized_block_sparsity"], "o-", color=col, label=f"target {float(S):.0%}: {sel['id']}")
    axes[2].axhline(float(S), color=col, ls=":", lw=1)
axes[2].set_ylim(0, 0.9); axes[2].set_title("P3 selected block sparsity"); axes[2].set_xlabel("block")
axes[2].legend(frameon=False, fontsize=8)
[a.xaxis.set_major_locator(MaxNLocator(integer=True)) for a in axes]
fig.tight_layout(); fig.savefig(FIG_DIR / "fig3_p3_sensitivity_allocation.png", dpi=150); plt.show()

Q3_SENS = load_json(OUT_DIR / "q3_sensitivity.json"); Q3_SEL = load_json(OUT_DIR / "q3_selection.json")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw=dict(width_ratios=[1.3, 1]))
M = np.array([Q3_SENS["probe_dnll"][str(b)] for b in Q3_BITS])
im = axes[0].imshow(np.clip(M, 1e-5, None), aspect="auto", cmap="Blues", norm=matplotlib.colors.LogNorm())
axes[0].set_yticks(range(len(Q3_BITS)), [f"{b}-bit" for b in Q3_BITS]); axes[0].set_xlabel("block")
axes[0].set_title("Q3 probe: calibration dNLL when only this block is quantized"); fig.colorbar(im, ax=axes[0], label="dNLL (log)")
for bu, col in (("4", C["uniform"]), ("3", C["adaptive"])):
    sel = next(c for c in Q3_SEL[bu]["candidates"] if c["id"] == Q3_SEL[bu]["selected"])
    axes[1].step(np.arange(L), sel["bits"], where="mid", color=col, label=f"INT{bu} budget: {sel['id']}")
axes[1].set_yticks(Q3_BITS); axes[1].set_xlabel("block"); axes[1].set_ylabel("assigned bits")
axes[1].set_title("Q3 selected bit allocation"); axes[1].legend(frameon=False, fontsize=8)
[a.xaxis.set_major_locator(MaxNLocator(integer=True)) for a in axes]
fig.tight_layout(); fig.savefig(FIG_DIR / "fig4_q3_sensitivity_allocation.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# Figure 5 — does a calibration / development criterion carry over to held-out quality?
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fam_col = {"P1": C["uniform"], "Q1": C["uniform"], "P2": C["aware"], "Q2": C["aware"], "P3": C["adaptive"],
           "Q3": C["adaptive"], "P2_24": C["struct"], "P3_unselected": C["cand"], "Q3_unselected": C["cand"]}
d = SUMMARY[SUMMARY.family != "baseline"]
for fam, sub in d.groupby("family"):   # circles = pruning, squares = quantization
    axes[0].scatter(sub.calib_dnll, sub.d_wt_nll, color=fam_col.get(fam, "#000"), s=45, label=fam,
                    marker="o" if fam.startswith("P") else "s", edgecolor="white", linewidth=1)
axes[0].set_xscale("symlog", linthresh=1e-3); axes[0].set_yscale("symlog", linthresh=1e-3)
axes[0].set_xlabel("calibration dNLL (C4 train, calibration IDs)"); axes[0].set_ylabel("held-out dNLL (WikiText-2)")
axes[0].set_title("Calibration criterion vs held-out"); axes[0].legend(frameon=False, fontsize=8)
rho = pd.Series(d.calib_dnll).rank().corr(pd.Series(d.d_wt_nll).rank())
axes[0].text(0.02, 0.95, f"Spearman rho = {rho:.2f} (n={len(d)})", transform=axes[0].transAxes, fontsize=9)
cand = SUMMARY[SUMMARY.family.isin(["P3", "P3_unselected", "Q3", "Q3_unselected"])]
for fam, sub in cand.groupby("family"):
    axes[1].scatter(sub.dev_nll, sub.wt_ppl, color=fam_col[fam], s=50, label=fam,
                    marker="o" if fam.startswith("P") else "s", edgecolor="white", linewidth=1)
axes[1].set_yscale("log"); axes[1].set_xlabel("development NLL (selection criterion)")
axes[1].set_ylabel("WikiText-2 PPL (never used for selection)"); axes[1].set_title("P3/Q3 candidates: dev vs held-out")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig5_calibration_transfer.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# Figure 6/7 — RQ1: magnitude vs activation; where does the error land?
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, S in zip(axes, (25, 50, 75)):
    k = f"P2_wanda_s{S}"
    if k not in OK: continue
    ss = R[k]["extra"]["score_stats"]
    for i, p in enumerate(PROJS):
        ys = [ss[f"{l}.{p}"]["mask_overlap_with_magnitude"] for l in range(L)]
        ax.plot(range(L), ys, lw=1.5, label=p, color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"][i])
    ax.set_title(f"sparsity {S}%: share of Wanda-pruned weights\nthat magnitude also prunes"); ax.set_xlabel("block")
axes[0].set_ylabel("mask overlap"); axes[2].legend(frameon=False, fontsize=7, ncol=2)
[a.xaxis.set_major_locator(MaxNLocator(integer=True)) for a in axes]
fig.tight_layout(); fig.savefig(FIG_DIR / "fig6_mask_overlap.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, k in zip(axes, ("P2_wanda_s50", "Q2_awq_int3")):
    if k not in OK: continue
    E = np.array([[R[k]["local_errors"][f"{l}.{p}"] for l in range(L)] for p in PROJS])
    im = ax.imshow(np.clip(E, 1e-6, None), aspect="auto", cmap="Blues", norm=matplotlib.colors.LogNorm())
    ax.set_yticks(range(len(PROJS)), PROJS); ax.set_xlabel("block")
    ax.set_title(f"{k}: relative layer-output error on calibration activations"); fig.colorbar(im, ax=ax)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig7_layer_error_maps.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# Theoretical (storage) versus measured (latency / allocated memory) savings
# ============================================================
res_cols = ["eligible_MB", "total_MB", "eff_bits_incl_meta", "compress_s", "prefill2048_ms", "decode_ms_tok", "peak_MB_prefill2048"]
print(main[res_cols].round(2).to_string())
base_lat = main.loc["FP16_baseline"]
print(f"\nMeasured prefill-2048 latency range across conditions: {main.prefill2048_ms.min():.1f}-{main.prefill2048_ms.max():.1f} ms "
      f"(baseline {base_lat.prefill2048_ms:.1f}); peak allocated MB range {main.peak_MB_prefill2048.min():.0f}-"
      f"{main.peak_MB_prefill2048.max():.0f}. All conditions execute DENSE FP16 kernels, so storage savings are packed-format "
      f"estimates, not measured speedups.")
for k in ("P2_wanda_s50", "P2_wanda_2of4", "Q2_awq_int4", "Q2_awq_int3"):
    if k in OK:
        st = R[k]["storage"]
        print(f"{k:>16}: dense FP16 {mb(st['eligible_dense_fp16_bytes']):.0f} MB | packed/sparse format {mb(st['eligible_bytes']):.0f} MB"
              + (f" | 2:4 compressed {mb(st['eligible_2of4_compressed_bytes']):.0f} MB" if "eligible_2of4_compressed_bytes" in st else ""))

In [ ]:
# ============================================================
# Qualitative: a PRE-FIXED set (the first 6 IDs of the HellaSwag diagnostic subset) + all
# baseline-correct -> compressed-wrong flips on the 512-ID subset
# ============================================================
SHOW = [k for k in ("FP16_baseline", "P1_mag_s50", "P2_wanda_s50", "P3_nonuni_s50", "P2_wanda_s75",
                    "Q1_rtn_int3", "Q2_awq_int3", "Q3_mixed_int3budget") if k in PE]
QUAL = []
for i in range(6):
    e = HS_ENC[i] if "HS_ENC" in globals() else None
    print(f"\n--- HellaSwag id {int(BASE_PE['hs_ids'][i])} (gold = {int(BASE_PE['hs_label'][i])}) ---")
    if e:
        print("context:", e["query"][:200])
        for j, c in enumerate(e["choices"]):
            print(f"  [{j}] {c[:90]}")
    for k in SHOW:
        sc = PE[k]["hs_ll"][i] / PE[k]["hs_nchar"][i]
        print(f"  {k:>22}: per-char loglik {np.round(sc, 3).tolist()} -> pick {int(sc.argmax())}"
              + ("  ✓" if sc.argmax() == BASE_PE["hs_label"][i] else "  ✗"))
        QUAL.append(dict(id=int(BASE_PE["hs_ids"][i]), condition=k, scores=sc.tolist(), pick=int(sc.argmax())))
save_json(QUAL, OUT_DIR / "qualitative_hellaswag.json")
bc = hs_correct(dict(ll=BASE_PE["hs_ll"][:HS_SUB], nchar=BASE_PE["hs_nchar"][:HS_SUB], label=BASE_PE["hs_label"][:HS_SUB]))
for k in SHOW[1:]:
    n = HS_SUB
    cc = hs_correct(dict(ll=PE[k]["hs_ll"][:n], nchar=PE[k]["hs_nchar"][:n], label=PE[k]["hs_label"][:n]))
    print(f"{k:>22}: baseline-right->wrong {int(((bc == 1) & (cc == 0)).sum())}, "
          f"baseline-wrong->right {int(((bc == 0) & (cc == 1)).sum())} (of {n})")

save_json(CHECKS, OUT_DIR / "checks.json")
print("\nSaved:", sorted(p.name for p in OUT_DIR.iterdir()))

## Done — what to send back
Download `t1_outputs/` (or just: `summary_all_conditions.csv`, `paired_bootstrap.csv`, `checks.json`, `figures/`,
`p3_selection.json`, `q3_selection.json`, `awq_fold_check.json`, `environment.json`) and share them — we'll go through the
results and the report story together before Task 2.